# UACB-604 · Lecture 2.5 — Finding and downloading sequencing data

Colab runs on a Linux computer, so we can use the same command-line tools you'd use on a server: `wget`, `curl`, `md5sum`, `zcat`.
Lines starting with `!` are **shell commands**; everything else is Python.

**You will:**
1. Climb the accession ladder: BioProject → BioSample → Experiment → Run
2. Ask ENA for download links, file sizes and checksums of a whole project
3. Estimate download times
4. Download a real run with `wget`, and check it with an **MD5 checksum**
5. See why a checksum catches even a one-byte error

In [ ]:
import io, json, time, hashlib, urllib.request
import pandas as pd

def get(url, tries=5):
    for attempt in range(tries):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "uacb604-course"})
            return urllib.request.urlopen(req, timeout=60).read().decode()
        except Exception as e:
            print(f"  (server busy: {e}; retrying)"); time.sleep(2 + 2 * attempt)
    raise RuntimeError("server not responding; try again in a minute")

ENA = "https://www.ebi.ac.uk/ena/portal/api/filereport"
FIELDS = "run_accession,experiment_accession,sample_accession,study_accession,instrument_model,library_layout,read_count,base_count,fastq_bytes,fastq_md5,fastq_ftp"

## 1. The accession ladder for one run

In [ ]:
run = pd.read_csv(io.StringIO(get(f"{ENA}?accession=SRR1039508&result=read_run&fields={FIELDS}")), sep="\t")
run.T

| Level | Prefix (NCBI / ENA / DDBJ) | What it is |
|---|---|---|
| BioProject | PRJNA / PRJEB / PRJDB | the whole study |
| BioSample | SAMN / SAMEA / SAMD | one biological sample |
| Experiment | SRX / ERX / DRX | one library prepared from a sample |
| Run | SRR / ERR / DRR | one sequencing run of that library → the FASTQ files |

In [ ]:
bp = json.loads(get("https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=bioproject&term=PRJNA229998&retmode=json"))["esearchresult"]["idlist"][0]
info = json.loads(get(f"https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=bioproject&id={bp}&retmode=json"))["result"][bp]
print(info["project_acc"], "|", info["project_title"])

## 2. Every run in the project

In [ ]:
proj = pd.read_csv(io.StringIO(get(f"{ENA}?accession=PRJNA229998&result=read_run&fields={FIELDS}")), sep="\t")
proj["GB"] = proj["fastq_bytes"].astype(str).str.split(";").apply(lambda xs: sum(int(x) for x in xs if x) / 1e9)
print(len(proj), "runs from", proj.sample_accession.nunique(), "samples")
proj[["run_accession", "sample_accession", "instrument_model", "read_count", "GB"]]

## 3. How long will the download take?
Network speeds are quoted in **bits** per second; file sizes in **bytes**. 1 byte = 8 bits.
$$\text{time (s)} = \frac{\text{size (bytes)} \times 8}{\text{speed (bits/s)}}$$

In [ ]:
total_GB = proj.GB.sum()
for mbps in (10, 100, 1000):
    seconds = total_GB * 1e9 * 8 / (mbps * 1e6)
    print(f"{total_GB:.1f} GB at {mbps:>4} Mbit/s: {seconds/60:6.0f} minutes")

## 4. Download a small real run with `wget`
A tiny phiX174 MiSeq run, **SRR41113426** (two files, about 1.9 MB together). First ask ENA for its links and checksums.

In [ ]:
small = pd.read_csv(io.StringIO(get(f"{ENA}?accession=SRR41113426&result=read_run&fields={FIELDS}")), sep="\t").iloc[0]
urls  = ["https://" + u for u in small.fastq_ftp.split(";")]
md5s  = small.fastq_md5.split(";")
for u, m in zip(urls, md5s):
    print(m, u)

In [ ]:
!wget -q --tries=5 https://ftp.sra.ebi.ac.uk/vol1/fastq/SRR411/026/SRR41113426/SRR41113426_1.fastq.gz
!wget -q --tries=5 https://ftp.sra.ebi.ac.uk/vol1/fastq/SRR411/026/SRR41113426/SRR41113426_2.fastq.gz
!ls -lh SRR41113426_*.fastq.gz
!zcat SRR41113426_1.fastq.gz | head -4

(The same files are also reachable with `ftp://ftp.sra.ebi.ac.uk/...`, which is what a graphical FTP client such as **FileZilla** uses. Colab works best with `https://`.)

## 5. Checksums: did the file arrive intact?
An **MD5 checksum** is a 32-character fingerprint computed from every byte of a file. ENA publishes the fingerprint; we compute our own and compare.

In [ ]:
!md5sum SRR41113426_1.fastq.gz SRR41113426_2.fastq.gz

In [ ]:
def md5_of(path):
    h = hashlib.md5()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

for u, expected in zip(urls, md5s):
    name = u.split("/")[-1]
    mine = md5_of(name)
    print(name, "OK ✓" if mine == expected else "MISMATCH ✗ (download again)")

### One changed byte changes everything

In [ ]:
data = open("SRR41113426_1.fastq.gz", "rb").read()
corrupt = bytearray(data); corrupt[1000] ^= 1          # flip a single bit in one byte
print("original :", hashlib.md5(data).hexdigest())
print("1 bit off:", hashlib.md5(bytes(corrupt)).hexdigest())

✏️ **Exercise 1.** MD5 fingerprints have 128 bits, so there are $2^{128} \approx 3.4\times10^{38}$ possible values. Use the birthday formula from Lecture 2.3 to estimate the chance that two *different* files among a billion ($10^9$) files share a fingerprint by accident.

✏️ **Exercise 2.** Write a loop that downloads both files of any run you choose (keep it under 20 MB!), checks both checksums, and prints a one-line report.

✏️ **Exercise 3.** NCBI's own tools do the same job: `prefetch SRR1039508` then `fasterq-dump SRR1039508` (from the **SRA Toolkit**). Read the SRA Toolkit documentation and write the two commands you would use to download only the first 10,000 reads.

---
**Further reading:** Buffalo, *Bioinformatics Data Skills*, ch. 6 (bioinformatics data: downloading, checksums) · Pevsner, ch. 2.